# Hybrid Document AI — OCR + KIE (production-grade, MLOps)
**Ứng tuyển AI Engineer @ VNPAY** — track *NLP · Computer Vision · MLOps*.

Pipeline hiểu chứng từ (receipt/invoice) theo **3-layer Banking/Enterprise** + **vòng đời MLOps khép kín**.
Notebook này chạy lại **toàn bộ từ đầu** trên máy này.

## 🔗 Links
- 🟢 **Live demo (HF Space):** https://huggingface.co/spaces/banhchungtuongot/hybrid-docai-demo
- 🐙 Code: https://github.com/NguyenDucAnforwork/hybrid-document-ai
- 🤗 Model (registry): https://huggingface.co/banhchungtuongot/hybrid-docai-kie
- 🤗 Dataset: https://huggingface.co/datasets/banhchungtuongot/hybrid-docai-receipts

## Kiến trúc
| Layer | Thành phần |
|---|---|
| **Processing** (multi-model) | Quality → OCR (RapidOCR/PP-OCR ONNX) → **KIE**: layout line-grouping → features → **calibrated sklearn classifier** → confidence router → VLM fallback |
| **Serving** | dynamic micro-batcher (chạy thật) · Triton/vLLM/KServe (artifact) · autoscale/scale-to-zero |
| **MLOps** | training-pipeline DAG (validate→train→eval-gate→register) · model registry có stage+lineage · monitoring+drift+alerts · chaos · DR |

## Dữ liệu & đánh giá (theo phản hồi reviewer)
- **Data thật:** SROIE 2019 (626 receipt scan thật) + **augmentation banking** (tối/nghiêng/mờ/motion/nhiễu/rách/fade/low-res/JPEG/perspective) + receipt **tiếng Việt**.
- **Metric mạnh:** exact-match **+ F1 + CER + ANLS + ECE (calibration)** + **robustness-curve** theo từng degradation.


## 0. Setup


In [ ]:
import os, sys, subprocess, json
os.environ.setdefault('DOCAI_WORKSPACE','/data/nvidia-ai-workspace')  # set TRƯỚC khi import docai
REPO=os.getcwd(); sys.path.insert(0,REPO); WS=os.environ['DOCAI_WORKSPACE']
PY=sys.executable; print('repo',REPO,'| ws',WS)


In [ ]:
# subprocess.run([PY,'-m','pip','install','-q','-r','requirements.txt'], check=True)
print('deps: dùng venv /data/nvidia-ai-workspace/venv (không cần torch; VLM ở api mode)')


## 1. Dữ liệu
### 1a. SROIE thật (clone mirror) + ingest → tokens+gold


In [ ]:
# git clone --depth1 https://github.com/zzzDavid/ICDAR-2019-SROIE  (đã có ở $WS/sroie_src)
subprocess.run([PY,'scripts/prepare_sroie.py','--n-test','80'])


### 1b. Synthetic (EN + tiếng Việt) cho 5 field + multilingual


In [ ]:
from docai.synth import generate
generate(f'{WS}/data/receipts',120,42); print('synthetic 120 ok')


### 1c. Augmentation banking-realistic (robustness)


In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from docai.augment import DEGRADATIONS, mixed_hard
img=cv2.imread(f'{WS}/data/sroie/test/images/'+os.listdir(f'{WS}/data/sroie/test/images')[0])
names=['dark','blur','motion_blur','rotate','noise','tear','fade','low_res','perspective']
fig,ax=plt.subplots(1,len(names),figsize=(20,4))
for a,n in zip(ax,names): a.imshow(cv2.cvtColor(DEGRADATIONS[n](img,0.6,1),cv2.COLOR_BGR2RGB)); a.set_title(n); a.axis('off')
plt.tight_layout(); plt.show()


## 2. Training pipeline DAG (MLOps) — validate → train → eval-gate → register
`mlops/pipeline.py`: caching/retry/resume + manifest lineage. Train kết hợp **SROIE thật + synthetic**, model có **calibration**.


In [ ]:
subprocess.run([PY,'-m','mlops.pipeline','--run-id','nb_demo','--version','v4'])


In [ ]:
# Model registry: stage (dev/staging/production/archived) + lineage
print(open(f'{WS}/models/registry.yaml').read())


In [ ]:
# Data validation gate (great-expectations style)
subprocess.run([PY,'-m','mlops.data_validation',f'{WS}/data/sroie/train/labels.json'])


## 3. Đánh giá mạnh
### 3a. Benchmark trên SROIE thật (clean): F1 + exact + ANLS + CER + eval-gate


In [ ]:
subprocess.run([PY,'scripts/run_benchmark.py','--data',f'{WS}/data/sroie/test','--f1-threshold','0.2'])


### 3b. Robustness curve — accuracy/ECE theo từng degradation (banking)


In [ ]:
subprocess.run([PY,'scripts/eval_robustness.py','--data',f'{WS}/data/sroie/test','--limit','30','--severity','0.6'])
# bảng -> docs/logs/robustness_*.md


## 4. Inference qua API (FastAPI TestClient)


In [ ]:
from fastapi.testclient import TestClient; from app.main import app
client=TestClient(app); print('health',client.get('/health').json())


In [ ]:
img=f'{WS}/data/sroie/test/images/'+os.listdir(f'{WS}/data/sroie/test/images')[0]
print(json.dumps(client.post('/documents/extract',files={'file':open(img,'rb')}).json(),indent=2,ensure_ascii=False))


In [ ]:
# /metrics + drift signals
m=client.get('/metrics').text
print('\n'.join(l for l in m.splitlines() if l.startswith(('documents_processed','vlm_fallback','human_review','field_confidence_count','input_blur_score_count'))))


## 5. MLOps độ bền — Chaos engineering
Bơm input OOD/blank/corrupt/tiny → assert hệ thống không crash, route human review (không phát data sai).


In [ ]:
subprocess.run([PY,'-m','mlops.chaos'])


## 6. Serving & Deployment
- **Dynamic micro-batcher** chạy thật: `docai/batcher.py` (test `tests/test_pipeline.py::test_batcher_groups`).
- **Live demo:** HF Space (Gradio) — link ở đầu notebook.
- Production target (artifact, swap-by-config): `deploy/kserve.yaml` (autoscale/scale-to-zero), `deploy/docker-compose.yml` (redis+minio+triton+vllm+prometheus), `mlops/kfp_pipeline.py` (Kubeflow), `monitoring/alerts.yaml`.
- Vận hành & DR: `docs/runbook-dr.md`; tổng quan MLOps: `docs/mlops.md`.


## 7. Kết quả & tài liệu
Số liệu thật xem output cell 3a/3b ở trên và `docs/logs/`. Tóm tắt + phân tích (gồm vì sao merchant_name khó, vì sao total-confidence không phân biệt đúng/sai, train/serve gap) trong **`README.md`** và **`docs/lessons-learned.md`**.

Tài liệu: `PLAN.md` · `docs/mlops.md` · `docs/lessons-learned.md` · `docs/debug-workflows.md` · `docs/reproduce.md` · `docs/runbook-dr.md` · `docs/logs/`.
